# MultiKAN Training for Massachusetts Buildings Segmentation

Semantic segmentation on the Massachusetts Buildings dataset with paired images and labels.

https://github.com/KindXiaoming/pykan/blob/master/kan/MultKAN.py

In [2]:
import os
import glob
import time
import csv
import random
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch.nn.functional as F
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
PROJECT_ROOT = os.path.abspath("..")
DATASET_ROOT = os.path.join(PROJECT_ROOT, "dataset", "archive")
DATASET_VARIANT = "tiff"  # "tiff" or "png"
MODEL_SAVE_DIR = os.path.join(PROJECT_ROOT, "MultiKAN", "models")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

IMG_SIZE = 256
NUM_CLASSES = 2
CLASS_NAMES = ["background", "building"]
IGNORE_INDEX = None  # set to 0 to ignore background in metrics

BATCH_SIZE = 1
LEARNING_RATE = 1e-4
EPOCHS = 50
USE_AMP = torch.cuda.is_available()
ACCUM_STEPS = 8
KAN_PIXEL_CHUNK = 8192
KAN_SYMBOLIC = False

KAN_WIDTH = [[3, 0], [16, 4], [16, 4], [1, 0]]
KAN_GRID = 5
KAN_K = 3
KAN_MULT_ARITY = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "train")
TRAIN_MASK_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "train_labels")
VAL_IMG_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "val")
VAL_MASK_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "val_labels")
TEST_IMG_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "test")
TEST_MASK_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "test_labels")

def _count_images(path, exts):
    count = 0
    for ext in exts:
        count += len(glob.glob(os.path.join(path, f"*.{ext}")))
    return count

exts = ["tif", "tiff"] if DATASET_VARIANT == "tiff" else ["png", "jpg", "jpeg"]

for p in [TRAIN_IMG_DIR, TRAIN_MASK_DIR, VAL_IMG_DIR, VAL_MASK_DIR, TEST_IMG_DIR, TEST_MASK_DIR]:
    if not os.path.isdir(p):
        raise FileNotFoundError(f"Missing directory: {p}")

print(f"Device: {DEVICE}")
print("Dataset variant:", DATASET_VARIANT)
print("Train images:", _count_images(TRAIN_IMG_DIR, exts), "Train labels:", _count_images(TRAIN_MASK_DIR, exts))
print("Val images:", _count_images(VAL_IMG_DIR, exts), "Val labels:", _count_images(VAL_MASK_DIR, exts))
print("Test images:", _count_images(TEST_IMG_DIR, exts), "Test labels:", _count_images(TEST_MASK_DIR, exts))

Device: cuda
Dataset variant: tiff
Train images: 137 Train labels: 137
Val images: 4 Val labels: 4
Test images: 10 Test labels: 10


/tmp/ipykernel_17688/1229701972.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


## Dataset Loader

In [4]:
class MassachusettsBuildingsDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=256, variant="tiff"):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.img_size = img_size
        self.exts = ["tif", "tiff"] if variant == "tiff" else ["png", "jpg", "jpeg"]

        self.pairs = self._collect_pairs()
        if len(self.pairs) == 0:
            raise RuntimeError(f"No image/mask pairs found in {image_dir} and {mask_dir}")

        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])
        self.mask_resize = transforms.Resize((img_size, img_size), interpolation=Image.NEAREST)

    def _collect_pairs(self):
        img_paths = []
        for ext in self.exts:
            img_paths.extend(sorted(self.image_dir.glob(f"*.{ext}")))
        mask_map = {}
        for ext in self.exts:
            for p in self.mask_dir.glob(f"*.{ext}"):
                mask_map[p.stem] = p
        pairs = []
        for img_path in img_paths:
            mask_path = mask_map.get(img_path.stem)
            if mask_path is not None:
                pairs.append((img_path, mask_path))
        return pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = Image.open(img_path).convert("L")
        mask = Image.open(mask_path)

        image = self.img_transform(image)
        mask = self.mask_resize(mask)
        mask = np.array(mask)
        if mask.ndim == 3:
            mask = mask[..., 0]
        mask = (mask > 127).astype(np.int64)
        mask = torch.from_numpy(mask)
        return image, mask

## MultiKAN Model (pykan MultKAN)

In [5]:
try:
    from kan import MultKAN
except Exception as exc:
    raise ImportError("Missing pykan. Install with: pip install pykan") from exc


class MultiKANSegmentation(nn.Module):
    def __init__(self, width, grid=5, k=3, mult_arity=2, pixel_chunk=8192, symbolic_enabled=False, device="cpu"):
        super().__init__()
        self.pixel_chunk = max(1, int(pixel_chunk))
        self.kan = MultKAN(
            width=width,
            grid=grid,
            k=k,
            mult_arity=mult_arity,
            symbolic_enabled=symbolic_enabled,
            device=device,
            auto_save=False,
            save_act=False,
        )

    def forward(self, x):
        b, c, h, w = x.shape
        device = x.device
        ys = torch.linspace(-1.0, 1.0, h, device=device)
        xs = torch.linspace(-1.0, 1.0, w, device=device)
        grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
        coords = torch.stack([grid_x, grid_y], dim=-1).view(-1, 2)
        coords = coords.repeat(b, 1)

        pix = x[:, :1, :, :].permute(0, 2, 3, 1).reshape(-1, 1)
        feats = torch.cat([coords, pix], dim=1)

        outputs = []
        for start in range(0, feats.shape[0], self.pixel_chunk):
            out_chunk = self.kan(feats[start : start + self.pixel_chunk])
            outputs.append(out_chunk)

        out = torch.cat(outputs, dim=0)
        out = out.view(b, h, w, -1).permute(0, 3, 1, 2)
        return out

## Losses, Optimizer, and Metrics

In [6]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, probs, targets):
        if probs.dim() == 4:
            probs = probs.squeeze(1)
        if targets.dim() == 4:
            targets = targets.squeeze(1)
        probs = probs.float()
        targets = targets.float()
        probs = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)
        intersection = (probs * targets).sum(dim=1)
        union = probs.sum(dim=1) + targets.sum(dim=1)
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()


class SegmentationMetrics:
    def __init__(self, eps=1e-7):
        self.eps = eps
        self.reset()

    def reset(self):
        self.tp = 0.0
        self.fp = 0.0
        self.fn = 0.0
        self.correct = 0.0
        self.total = 0.0

    def update(self, probs, targets):
        if probs.dim() == 4:
            probs = probs.squeeze(1)
        if targets.dim() == 4:
            targets = targets.squeeze(1)
        preds = (probs >= 0.5).long()
        targets = targets.long()
        self.correct += (preds == targets).sum().item()
        self.total += targets.numel()
        self.tp += ((preds == 1) & (targets == 1)).sum().item()
        self.fp += ((preds == 1) & (targets == 0)).sum().item()
        self.fn += ((preds == 0) & (targets == 1)).sum().item()

    def compute(self):
        precision = self.tp / (self.tp + self.fp + self.eps)
        recall = self.tp / (self.tp + self.fn + self.eps)
        f1 = 2 * precision * recall / (precision + recall + self.eps)
        iou = self.tp / (self.tp + self.fp + self.fn + self.eps)
        dice = 2 * self.tp / (2 * self.tp + self.fp + self.fn + self.eps)
        accuracy = self.correct / max(self.total, 1)
        return {
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "iou": float(iou),
            "dice": float(dice),
            "dice_loss": float(1.0 - dice),
        }


model = MultiKANSegmentation(
    width=KAN_WIDTH,
    grid=KAN_GRID,
    k=KAN_K,
    mult_arity=KAN_MULT_ARITY,
    pixel_chunk=KAN_PIXEL_CHUNK,
    symbolic_enabled=KAN_SYMBOLIC,
    device=DEVICE,
).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)

criterion_bce = nn.BCEWithLogitsLoss()
criterion_dice = DiceLoss()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 9,848


## DataLoaders

In [7]:
num_workers = min(4, os.cpu_count() or 1)

train_dataset = MassachusettsBuildingsDataset(
    TRAIN_IMG_DIR,
    TRAIN_MASK_DIR,
    img_size=IMG_SIZE,
    variant=DATASET_VARIANT,
)
val_dataset = MassachusettsBuildingsDataset(
    VAL_IMG_DIR,
    VAL_MASK_DIR,
    img_size=IMG_SIZE,
    variant=DATASET_VARIANT,
)
test_dataset = MassachusettsBuildingsDataset(
    TEST_IMG_DIR,
    TEST_MASK_DIR,
    img_size=IMG_SIZE,
    variant=DATASET_VARIANT,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Train samples: 137
Val samples: 4
Test samples: 10


In [8]:
def format_metrics(prefix, metrics):
    return (
        f"{prefix} acc={metrics['accuracy']:.4f} "
        f"prec={metrics['precision']:.4f} recall={metrics['recall']:.4f} "
        f"f1={metrics['f1']:.4f} dice={metrics['dice']:.4f} "
        f"iou={metrics['iou']:.4f} dice_loss={metrics['dice_loss']:.4f}"
    )


def run_epoch(loader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    metrics = SegmentationMetrics()
    total_loss = 0.0
    total_steps = len(loader)

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    with torch.set_grad_enabled(is_train):
        for step, (images, masks) in enumerate(loader, start=1):
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True).float()
            if masks.dim() == 3:
                masks = masks.unsqueeze(1)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(images)
                loss_bce = criterion_bce(logits, masks)
                probs = torch.sigmoid(logits)
                loss_dice = criterion_dice(probs, masks)
                loss = loss_bce + loss_dice

            if is_train:
                scaled_loss = loss / ACCUM_STEPS
                scaler.scale(scaled_loss).backward()
                if step % ACCUM_STEPS == 0 or step == total_steps:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

            total_loss += loss.item() * images.size(0)
            metrics.update(probs, masks)

    avg_loss = total_loss / max(len(loader.dataset), 1)
    return avg_loss, metrics.compute()

## Training Loop

In [8]:
history = {"train_loss": [], "val_loss": [], "train_metrics": [], "val_metrics": []}
best_val_loss = float("inf")
best_model_path = os.path.join(MODEL_SAVE_DIR, "multikan_best.pth")
train_metrics_path = os.path.join(MODEL_SAVE_DIR, "metrics_train_val.csv")
best_train_loss = float("inf")
best_train_epoch = 0

train_start = time.time()
print("Training start:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(train_start)))

with open(train_metrics_path, "w", newline="") as metrics_file:
    writer = csv.writer(metrics_file)
    writer.writerow(["epoch", "split", "loss", "accuracy", "precision", "recall", "f1", "dice", "iou", "dice_loss"])

    for epoch in range(EPOCHS):
        epoch_start = time.time()
        train_loss, train_metrics = run_epoch(train_loader, model, optimizer=optimizer)
        val_loss, val_metrics = run_epoch(val_loader, model, optimizer=None)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_metrics"].append(train_metrics)
        history["val_metrics"].append(val_metrics)

        if train_loss < best_train_loss:
            best_train_loss = train_loss
            best_train_epoch = epoch + 1

        writer.writerow([
            epoch + 1,
            "train",
            train_loss,
            train_metrics["accuracy"],
            train_metrics["precision"],
            train_metrics["recall"],
            train_metrics["f1"],
            train_metrics["dice"],
            train_metrics["iou"],
            train_metrics["dice_loss"],
        ])
        writer.writerow([
            epoch + 1,
            "val",
            val_loss,
            val_metrics["accuracy"],
            val_metrics["precision"],
            val_metrics["recall"],
            val_metrics["f1"],
            val_metrics["dice"],
            val_metrics["iou"],
            val_metrics["dice_loss"],
        ])
        metrics_file.flush()

        print(f"Epoch {epoch + 1}/{EPOCHS} | train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
        print(format_metrics("Train", train_metrics))
        print(format_metrics("Val", val_metrics))
        print(f"Epoch time: {time.time() - epoch_start:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": best_val_loss,
                },
                best_model_path,
            )
            print(f"Saved best model: {best_model_path}")

train_end = time.time()
print("Training end:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(train_end)))
print(f"Total training time: {train_end - train_start:.1f}s")
print(f"Best train epoch: {best_train_epoch} (train_loss={best_train_loss:.4f})")
print(f"Train/val metrics saved to {train_metrics_path}")

Training start: 2026-05-29 19:51:22


/tmp/ipykernel_12512/1317985534.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Epoch 1/50 | train_loss=1.4904 val_loss=1.5118
Train acc=0.6708 prec=0.1412 recall=0.2940 f1=0.1908 dice=0.1908 iou=0.1054 dice_loss=0.8092
Val acc=0.8841 prec=0.2958 recall=0.0478 f1=0.0823 dice=0.0823 iou=0.0429 dice_loss=0.9177
Epoch time: 7.0s
Saved best model: /media/aejaz/New_Volume/Projects/Massachusetts/MultiKAN/models/multikan_best.pth
Epoch 2/50 | train_loss=1.4846 val_loss=1.5055
Train acc=0.8520 prec=0.1439 recall=0.0245 f1=0.0418 dice=0.0418 iou=0.0214 dice_loss=0.9582
Val acc=0.8913 prec=0.0000 recall=0.0000 f1=0.0000 dice=0.0000 iou=0.0000 dice_loss=1.0000
Epoch time: 6.7s
Saved best model: /media/aejaz/New_Volume/Projects/Massachusetts/MultiKAN/models/multikan_best.pth
Epoch 3/50 | train_loss=1.4788 val_loss=1.4994
Train acc=0.8680 prec=0.0904 recall=0.0000 f1=0.0000 dice=0.0000 iou=0.0000 dice_loss=1.0000
Val acc=0.8913 prec=0.0000 recall=0.0000 f1=0.0000 dice=0.0000 iou=0.0000 dice_loss=1.0000
Epoch time: 6.7s
Saved best model: /media/aejaz/New_Volume/Projects/Massach

## Test Evaluation

In [9]:
if os.path.isfile(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best model from {best_model_path}")

test_loss, test_metrics = run_epoch(test_loader, model, optimizer=None)
print(f"Test loss: {test_loss:.4f}")
print(format_metrics("Test", test_metrics))

test_metrics_path = os.path.join(MODEL_SAVE_DIR, "metrics_test.csv")
with open(test_metrics_path, "w", newline="") as metrics_file:
    writer = csv.writer(metrics_file)
    writer.writerow(["split", "loss", "accuracy", "precision", "recall", "f1", "dice", "iou", "dice_loss"])
    writer.writerow([
        "test",
        test_loss,
        test_metrics["accuracy"],
        test_metrics["precision"],
        test_metrics["recall"],
        test_metrics["f1"],
        test_metrics["dice"],
        test_metrics["iou"],
        test_metrics["dice_loss"],
    ])
print(f"Test metrics saved to {test_metrics_path}")

Loaded best model from /media/aejaz/New_Volume/Projects/Massachusetts/MultiKAN/models/multikan_best.pth


/tmp/ipykernel_12512/1889342327.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(best_model_path, map_location=DEVICE)
/tmp/ipykernel_12512/131798

Test loss: 1.3061
Test acc=0.8137 prec=0.0000 recall=0.0000 f1=0.0000 dice=0.0000 iou=0.0000 dice_loss=1.0000
Test metrics saved to /media/aejaz/New_Volume/Projects/Massachusetts/MultiKAN/models/metrics_test.csv


## Save Final Model

In [10]:
final_model_path = os.path.join(MODEL_SAVE_DIR, "multikan_final.pth")
torch.save(model.state_dict(), final_model_path)
print(f"Final model saved to {final_model_path}")

Final model saved to /media/aejaz/New_Volume/Projects/Massachusetts/MultiKAN/models/multikan_final.pth


In [20]:
# ============================================================
# MultiKAN – Export 600 DPI comparison images
# ============================================================
# Add this as a NEW CELL at the bottom of train_multi-kan.ipynb
# Run all cells EXCEPT Training Loop & Test Evaluation, then run this.
#
# Must already be in scope:
#   test_dataset, DEVICE, MODEL_SAVE_DIR
# ============================================================

from pathlib import Path
from PIL import Image, ImageDraw
import numpy as np
import torch
import torch.nn as nn

EXPORT_DPI  = 600
EXPORT_SIZE = 2048
OUTPUT_DIR  = Path(MODEL_SAVE_DIR).parent / "comparison_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from kan import MultKAN
except Exception as exc:
    raise ImportError("Missing pykan. Install with: pip install pykan") from exc


class MultiKANSegmentation(nn.Module):
    def __init__(self, width, grid=5, k=3, mult_arity=2,
                 pixel_chunk=8192, symbolic_enabled=False, device="cpu"):
        super().__init__()
        self.pixel_chunk = max(1, int(pixel_chunk))
        self.kan = MultKAN(
            width=width, grid=grid, k=k, mult_arity=mult_arity,
            symbolic_enabled=symbolic_enabled, device=device,
            auto_save=False, save_act=False,
        )

    def forward(self, x):
        b, c, h, w = x.shape
        device = x.device
        ys = torch.linspace(-1.0, 1.0, h, device=device)
        xs = torch.linspace(-1.0, 1.0, w, device=device)
        grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
        coords = torch.stack([grid_x, grid_y], dim=-1).view(-1, 2)
        coords = coords.repeat(b, 1)
        pix    = x[:, :1, :, :].permute(0, 2, 3, 1).reshape(-1, 1)
        feats  = torch.cat([coords, pix], dim=1)
        outputs = []
        for start in range(0, feats.shape[0], self.pixel_chunk):
            outputs.append(self.kan(feats[start: start + self.pixel_chunk]))
        out = torch.cat(outputs, dim=0)
        return out.view(b, h, w, -1).permute(0, 3, 1, 2)


# ── Load model ────────────────────────────────────────────────
export_model = MultiKANSegmentation(
    width=KAN_WIDTH, grid=KAN_GRID, k=KAN_K,
    mult_arity=KAN_MULT_ARITY, pixel_chunk=KAN_PIXEL_CHUNK,
    symbolic_enabled=KAN_SYMBOLIC, device=DEVICE.type,
).to(DEVICE)

best_model_path = os.path.join(MODEL_SAVE_DIR, "multikan_best.pth")
if Path(best_model_path).is_file():
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    export_model.load_state_dict(state_dict)
    print(f"Loaded best model from {best_model_path}")
else:
    print("[WARN] multikan_best.pth not found — using current weights")
    export_model.load_state_dict(model.state_dict())

# ── Inference ─────────────────────────────────────────────────
export_model.eval()
image_tensor, mask_tensor = test_dataset[0]
image_batch = image_tensor.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    logits = export_model(image_batch)

if isinstance(logits, (tuple, list)):
    logits = logits[0]

probs    = torch.sigmoid(logits)
prob_map = probs[0, 0]

# Adaptive threshold: use 0.5 if any pixel reaches it, else use mean
THRESHOLD = 0.5
if (prob_map >= THRESHOLD).sum().item() == 0:
    THRESHOLD = float(prob_map.mean())
    print(f"[WARN] No pixels >= 0.5 — model may not have converged well.")
    print(f"[INFO] Using adaptive threshold = {THRESHOLD:.4f}")

pred_mask = (prob_map >= THRESHOLD).long()
print(f"[INFO] Predicted foreground: {pred_mask.sum().item()} / {pred_mask.numel()} pixels")

# ── Helpers ───────────────────────────────────────────────────
def _to_rgb_image(tensor):
    t = tensor.detach().cpu().clamp(0, 1)
    if t.shape[0] == 1:
        t = t.repeat(3, 1, 1)
    return (t.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)

def _to_bw_mask(tensor):
    array = tensor.detach().cpu().squeeze().numpy()
    binary = array > 0.5 if array.max() <= 1 else array > 0
    return (binary.astype(np.uint8) * 255)

def _save_png_600dpi(array, path, resample):
    image = Image.fromarray(array)
    if image.size != (EXPORT_SIZE, EXPORT_SIZE):
        image = image.resize((EXPORT_SIZE, EXPORT_SIZE), resample=resample)
    image.save(path, dpi=(EXPORT_DPI, EXPORT_DPI))
    return image

def _with_title(image, title):
    rgb          = image.convert("RGB")
    title_height = 128
    canvas       = Image.new("RGB", (rgb.width, rgb.height + title_height), "white")
    canvas.paste(rgb, (0, title_height))
    draw = ImageDraw.Draw(canvas)
    draw.rectangle((0, 0, rgb.width, title_height), fill="black")
    draw.text((40, 42), title, fill="white")
    return canvas

# ── Save ──────────────────────────────────────────────────────
stem = "test_sample"
if hasattr(test_dataset, "pairs") and test_dataset.pairs:
    stem = Path(test_dataset.pairs[0][0]).stem

input_image        = _save_png_600dpi(_to_rgb_image(image_tensor), OUTPUT_DIR / f"{stem}_input_600dpi.png",        Image.Resampling.BICUBIC)
ground_truth_image = _save_png_600dpi(_to_bw_mask(mask_tensor),    OUTPUT_DIR / f"{stem}_ground_truth_600dpi.png", Image.Resampling.NEAREST)
prediction_image   = _save_png_600dpi(_to_bw_mask(pred_mask),      OUTPUT_DIR / f"{stem}_prediction_600dpi.png",   Image.Resampling.NEAREST)

panels = [
    _with_title(input_image,        "Input"),
    _with_title(ground_truth_image, "Ground Truth"),
    _with_title(prediction_image,   "Prediction"),
]
comparison = Image.new("RGB", (sum(p.width for p in panels), max(p.height for p in panels)), "white")
x = 0
for panel in panels:
    comparison.paste(panel, (x, 0))
    x += panel.width

comparison_path = OUTPUT_DIR / f"{stem}_comparison_600dpi.png"
comparison.save(comparison_path, dpi=(EXPORT_DPI, EXPORT_DPI))

print(f"Saved input:        {OUTPUT_DIR / (stem + '_input_600dpi.png')}")
print(f"Saved ground truth: {OUTPUT_DIR / (stem + '_ground_truth_600dpi.png')}")
print(f"Saved prediction:   {OUTPUT_DIR / (stem + '_prediction_600dpi.png')}")
print(f"Saved comparison:   {comparison_path}")

Loaded best model from /media/aejaz/New Volume/Projects/Massachusetts/MultiKAN/models/multikan_best.pth
[WARN] No pixels >= 0.5 — model may not have converged well.
[INFO] Using adaptive threshold = 0.1851
[INFO] Predicted foreground: 37191 / 65536 pixels
Saved input:        /media/aejaz/New Volume/Projects/Massachusetts/MultiKAN/comparison_outputs/22828930_15_input_600dpi.png
Saved ground truth: /media/aejaz/New Volume/Projects/Massachusetts/MultiKAN/comparison_outputs/22828930_15_ground_truth_600dpi.png
Saved prediction:   /media/aejaz/New Volume/Projects/Massachusetts/MultiKAN/comparison_outputs/22828930_15_prediction_600dpi.png
Saved comparison:   /media/aejaz/New Volume/Projects/Massachusetts/MultiKAN/comparison_outputs/22828930_15_comparison_600dpi.png


In [10]:
# ── DEBUG: paste this BEFORE the pred_mask line and run ──────
with torch.no_grad():
    logits = export_model(image_batch)

print("logits shape :", logits.shape)
print("logits min   :", logits.min().item())
print("logits max   :", logits.max().item())
print("logits mean  :", logits.mean().item())

probs = torch.sigmoid(logits)
print("probs min    :", probs.min().item())
print("probs max    :", probs.max().item())
print("probs mean   :", probs.mean().item())

pred_05 = (probs[0, 0] >= 0.5).sum().item()
pred_03 = (probs[0, 0] >= 0.3).sum().item()
pred_01 = (probs[0, 0] >= 0.1).sum().item()
print(f"pixels >= 0.5: {pred_05}")
print(f"pixels >= 0.3: {pred_03}")
print(f"pixels >= 0.1: {pred_01}")

# Also check the mask
print("mask unique values:", torch.unique(mask_tensor))
print("mask shape:", mask_tensor.shape)

logits shape : torch.Size([1, 1, 256, 256])
logits min   : -1.8407313823699951
logits max   : -1.3233274221420288
logits mean  : -1.4853744506835938
probs min    : 0.136964812874794
probs max    : 0.21026523411273956
probs mean   : 0.18508988618850708
pixels >= 0.5: 0
pixels >= 0.3: 0
pixels >= 0.1: 65536
mask unique values: tensor([0, 1])
mask shape: torch.Size([256, 256])
